In [1]:
from pathlib import Path

import pandas as pd


CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "_data").exists() else CWD.parents[2]

RAW_DIR = PROJECT_ROOT / "_data" / "01_raw"
OUTPUT_DIR = PROJECT_ROOT / "_data" / "02_interim" / "260429 name changed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMN_RENAME_MAPS = {
    "Membership.csv": {
        "user_no": "USER_KEY",
        "product_cd": "product_code",
        "amount": "price",
        "concurrent_streams": "max_screen",
        "promotion_yn": "is_promotion",
        "repurchase": "is_repurchase",
    },
    "Movie_Master.csv": {
        "MOVIE_ID": "MOVIE_NUM",
        "TITLE": "movie_title",
        "RELEASE_MONTH": "ott_release_month",
    },
    "User_Mapping.csv": {
        "uid": "USER_KEY",
        "USER_ID": "USER_NUM",
    },
    "View_History.csv": {
        "USER_ID": "USER_NUM",
        "MOVIE_ID": "MOVIE_NUM",
        "DURATION": "watch_time(min)",
        "WATCH_DAY": "watch_day",
        "WATCH_SEQ": "watch_seq",
    },
}

membership = pd.read_csv(RAW_DIR / "Membership.csv")
membership = membership.rename(columns=COLUMN_RENAME_MAPS["Membership.csv"])
membership.to_csv(OUTPUT_DIR / "Membership.csv", index=False, encoding="utf-8", lineterminator="\n")

membership.head()

,USER_KEY,product_code,price,billing_method,max_screen,is_promotion,is_churn_prevented,is_repurchase,payment_device,is_user_verified,gender,age,reg_date,reg_hour,end_date
0,7a6960912bebe03c6e4c770eb1aa91329c3497f18f90ca...,pk_1489,100.00,134,4.0,O,NaN,NaN,pc,Y,F,20.0,2021-03-14,20,2021-04-14
1,4ec765db76545c1d6dda9f421590bf9d02f584009f8d92...,pk_1487,100.00,190,1.0,O,O,NaN,pc,Y,F,25.0,2021-03-09,14,2021-04-09
2,4f86d917c53cb6bd8949f76dba7260311e8c1748748a02...,pk_1487,100.00,132,1.0,O,NaN,NaN,android,NaN,F,55.0,2021-03-09,22,2021-04-09
3,445fb8813626d3d49b94b5be58cd76d80ed31fa94f8372...,pk_1508,9.99,140,1.0,NaN,NaN,O,ios,N,N,40.0,2021-03-09,10,2021-04-10
4,01b16f9f7ff29b48b1ee0d1a89d1eb9662474e5eedb8c2...,pk_1488,100.00,180,2.0,O,NaN,O,android,N,F,20.0,2021-03-09,2,2021-04-09


In [2]:
movie_master = pd.read_csv(RAW_DIR / "Movie_Master.csv")
movie_master = movie_master.rename(columns=COLUMN_RENAME_MAPS["Movie_Master.csv"])
movie_master.to_csv(OUTPUT_DIR / "Movie_Master.csv", index=False, encoding="utf-8", lineterminator="\n")

movie_master.head()

,MOVIE_NUM,movie_title,ott_release_month
0,0,걸어서하늘까지(1992),202103
1,1,너와극장에서,201807
2,2,가려진시간[가치봄],202010
3,3,그링고,202003
4,4,스위치(2010),202010


In [3]:
user_mapping = pd.read_csv(RAW_DIR / "User_Mapping.csv")
user_mapping = user_mapping.rename(columns=COLUMN_RENAME_MAPS["User_Mapping.csv"])
user_mapping.to_csv(OUTPUT_DIR / "User_Mapping.csv", index=False, encoding="utf-8", lineterminator="\n")

user_mapping.head()

,USER_KEY,USER_NUM
0,ce6c11de3c6c21a993965a45943bce46f068b56869feca...,0
1,a314c8a65b0fd91e1889c1debaa21e5bb6330560ee9721...,1
2,6652e2597875d4f34b48a0610c035521f2aea2f8abc5f3...,2
3,873fb3eddac14ca6123810dd72a4195915afacaaecb073...,3
4,8668c3dfce041e29bc5b80fe33fb8754718587a7537db1...,4


In [4]:
view_history = pd.read_csv(RAW_DIR / "View_History.csv")
view_history = view_history.rename(columns=COLUMN_RENAME_MAPS["View_History.csv"])
view_history.to_csv(OUTPUT_DIR / "View_History.csv", index=False, encoding="utf-8", lineterminator="\n")

view_history.head()

,USER_NUM,MOVIE_NUM,watch_time(min),watch_day,watch_seq
0,0,4243,1,20210314,1
1,0,4243,111,20210314,2
2,0,1860,101,20210319,1
3,0,2170,1,20210325,1
4,0,6031,100,20210326,1


In [5]:
description = pd.read_excel(RAW_DIR / "Description.xlsx", sheet_name="Spec")
description = description.copy()

membership_rename_map = COLUMN_RENAME_MAPS["Membership.csv"]
description["Name"] = description["Name"].replace(membership_rename_map)

description.to_excel(OUTPUT_DIR / "Description.xlsx", sheet_name="Spec", index=False)

description.head()

,Data,Name,Description,Example
0,Membership,USER_KEY,User 식별값,9c1c04380d3ec71c9ea55cb99ad803ab7c0037a3482b9b...
1,Membership,product_code,가입상품코드,pk_1487
2,Membership,price,실결제 금액,100
3,Membership,billing_method,결제 수단,190
4,Membership,max_screen,동시 시청 가능수,1


In [6]:
rename_summary = []

for file_name, rename_map in COLUMN_RENAME_MAPS.items():
    raw_columns = pd.read_csv(RAW_DIR / file_name, nrows=0).columns.tolist()
    changed_columns = pd.read_csv(OUTPUT_DIR / file_name, nrows=0).columns.tolist()

    for before, after in zip(raw_columns, changed_columns):
        rename_summary.append(
            {
                "file": file_name,
                "before": before,
                "after": after,
                "changed": before != after,
            }
        )

raw_description = pd.read_excel(RAW_DIR / "Description.xlsx", sheet_name="Spec")
changed_description = pd.read_excel(OUTPUT_DIR / "Description.xlsx", sheet_name="Spec")

for before, after in zip(raw_description["Name"], changed_description["Name"]):
    rename_summary.append(
        {
            "file": "Description.xlsx",
            "before": before,
            "after": after,
            "changed": before != after,
        }
    )

rename_summary_df = pd.DataFrame(rename_summary)
rename_summary_df

,file,before,after,changed
0,Membership.csv,user_no,USER_KEY,True
1,Membership.csv,product_cd,product_code,True
2,Membership.csv,amount,price,True
3,Membership.csv,billing_method,billing_method,False
4,Membership.csv,concurrent_streams,max_screen,True
5,Membership.csv,promotion_yn,is_promotion,True
6,Membership.csv,is_churn_prevented,is_churn_prevented,False
7,Membership.csv,repurchase,is_repurchase,True
8,Membership.csv,payment_device,payment_device,False
9,Membership.csv,is_user_verified,is_user_verified,False
